> **SmartVal AI's problem:** A real-estate analytics startup needs to predict median house values for California districts to power automated property appraisals. Their model must achieve MAE ≤ $40k — that is the regulatory threshold: exceed it and the regulator won't approve automated appraisals, ending the product. The dataset is 20,640 California census districts with 8 features (median income, house age, rooms, occupancy, latitude, longitude) and one target: median house value in $100k units. But good data alone isn't enough — _which_ ML technique to use, _how_ to train it, and _what goes wrong_ at each step all have concrete answers that this notebook walks through. Six building blocks, one dataset, one regulatory target: by the end, SmartVal will have a model recommendation, a minimum data requirement, and the evidence to back both up.


# ML Basics: Regression, Classification, and Generalisation

| Part | Concept                       | Key question answered                                         |
| ---- | ----------------------------- | ------------------------------------------------------------- |
| 1    | Linear regression             | Predict house price from 8 features; what's our baseline MAE? |
| 2    | Gradient descent from scratch | Can we train a linear model manually and match sklearn?       |
| 3    | Loss functions                | Does MSE or MAE work better when districts are outliers?      |
| 4    | Regularisation                | Ridge vs. Lasso: which features matter most?                  |
| 5    | Classification                | Can we classify "high value" districts (>$300k median)?       |
| 6    | Overfitting                   | What happens when we train on only 50 samples?                |

---

> **Dataset:** California Housing (`sklearn.datasets.fetch_california_housing`). No download needed. 8 features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude. Target: median house value ($100k units).


In [ ]:
#  Dependencies and Data 
import subprocess, sys

for pkg in ["numpy", "matplotlib", "scikit-learn", "seaborn"]:
    try:
        __import__(pkg.replace("-", "_").split("==")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    confusion_matrix,
)

np.random.seed(42)

# Load California Housing dataset
housing = fetch_california_housing()
X_raw, y = housing.data, housing.target
feature_names = housing.feature_names

print(f"Dataset: {X_raw.shape[0]:,} districts × {X_raw.shape[1]} features")
print(f"Target: median house value ($100k units)")
print(f"Features: {list(feature_names)}")
print()
print(f"Price range: ${y.min():.2f}00k – ${y.max():.2f}00k")
print(f"Mean price:  ${y.mean():.2f}00k  (≈ ${y.mean()*100:.0f}k)")

In [ ]:
#  Train/val/test split + feature scaling 
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  |  Val: {X_val_s.shape}  |  Test: {X_test_s.shape}")
print()
print("SmartVal's pipeline:")
print("  1. Standardise features (mean=0, std=1) so all features are comparable")
print("  2. Train on 66% of data, validate on 17%, hold 17% for final evaluation")
print("  3. Never touch the test set until the very end")

---

## Part 1 — Linear Regression: Fitting a Straight Line

SmartVal's first question is the most basic one: _can 8 census numbers — income, house age, rooms, occupancy, latitude, longitude — actually predict house price well enough for a regulator to approve?_ Before reaching for anything sophisticated, let's try the simplest model that could possibly work: a weighted sum of those 8 features.

Linear regression predicts: $\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_8 x_8 + b$

Each weight $w_i$ says "how much does feature $i$ contribute to the price?" The bias $b$ is a baseline price. We minimise the Mean Squared Error: $\text{MSE} = \frac{1}{n}\sum(\hat{y}_i - y_i)^2$

#### #### Predict first

Before fitting, predict: which single feature will have the highest positive weight in the linear model?

1. **MedInc** (median income) — richer districts have higher house prices
2. **Latitude** — southern California is more expensive than northern
3. **AveRooms** — more rooms = higher price

Make your prediction, then run the fit.


_(The loss landscape visualising what gradient descent minimises is shown before the gradient descent code in Part 2.)_


In [ ]:
#  Part 1: Linear regression 
lr = LinearRegression()
lr.fit(X_train_s, y_train)

train_pred = lr.predict(X_train_s)
val_pred = lr.predict(X_val_s)

train_mae = mean_absolute_error(y_train, train_pred) * 100  # convert to $k
val_mae = mean_absolute_error(y_val, val_pred) * 100

print(f"Linear regression results:")
print(f"  Train MAE: ${train_mae:.1f}k")
print(f"  Val MAE:   ${val_mae:.1f}k  (target: ≤$40k)")
print()
print("Feature weights (standardised → comparable):")
for name, w in sorted(
    zip(feature_names, lr.coef_), key=lambda x: abs(x[1]), reverse=True
):
    bar = "+" * int(abs(w) * 10) if w > 0 else "-" * int(abs(w) * 10)
    print(f"  {name:12s}: {w:+.4f}  {bar[:30]}")
print(f"\n  Intercept (bias): {lr.intercept_:.4f}")
print()
print("Prediction check: answer depends on actual run — the model tells you!")

#  Feature importance chart 
sorted_pairs = sorted(zip(feature_names, lr.coef_), key=lambda x: x[1])
names_s, coefs_s = zip(*sorted_pairs)
colors = ["steelblue" if c > 0 else "coral" for c in coefs_s]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(names_s, coefs_s, color=colors)
ax.axvline(0, color="white", lw=1.0)
ax.set_xlabel("Coefficient (standardised features)")
ax.set_title(
    "Which features drive California house prices? (positive=↑price, negative=↓price)"
)
plt.tight_layout()
plt.show()

In [ ]:
#  Part 1: Actual vs. predicted scatter 
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_val * 100, val_pred * 100, alpha=0.3, s=15, c="steelblue")
lo, hi = 0, 600
ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual price ($k)")
ax.set_ylabel("Predicted price ($k)")
ax.set_title(f"Linear regression — Val MAE = ${val_mae:.1f}k")
ax.legend()
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
plt.tight_layout()
plt.show()

# District 42 example
d42 = X_val_s[42].reshape(1, -1)
pred_42 = lr.predict(d42)[0] * 100
true_42 = y_val[42] * 100
print(f"SmartVal example — District 42:")
print(
    f"  Predicted: ${pred_42:.1f}k  |  Actual: ${true_42:.1f}k  |  Error: ${abs(pred_42-true_42):.1f}k"
)

#### What just happened — and what's missing

Linear regression found 8 weights that minimise squared error across 13,200 training districts — **MedInc dominated**, confirming that income is the strongest predictor of California house prices. The val MAE above is SmartVal's first concrete answer: a simple weighted sum of census features lands at or near the regulatory $40k threshold.

**What's missing:** `sklearn.LinearRegression` found those weights in one call, but _how_? For a linear model there's an analytical matrix formula — but the same approach can't train a neural network with millions of parameters. The universal training algorithm that scales to any model size is what Part 2 builds from scratch.

---

#### #### Your turn — cost of geographic features

The model above uses all 8 features including Latitude and Longitude. Drop those two geographic features and see how much the val MAE increases — this tells you whether geographic information is worth collecting.


In [ ]:
#  #### Your turn: cost of dropping geographic features 
# # CHANGE drop_indices to explore other subsets (e.g. [2,3]=AveRooms/AveBedrms)
drop_indices = [6, 7]  # # CHANGE ME — 6=Latitude, 7=Longitude

keep_mask = [i for i in range(X_train_s.shape[1]) if i not in drop_indices]
lr_sub = LinearRegression()
lr_sub.fit(X_train_s[:, keep_mask], y_train)
sub_val_mae = mean_absolute_error(y_val, lr_sub.predict(X_val_s[:, keep_mask])) * 100

print(f"Dropped indices {drop_indices}  |  Kept: {keep_mask}")
print(f"  Val MAE: ${sub_val_mae:.1f}k  (full-feature baseline: ${val_mae:.1f}k)")
delta = sub_val_mae - val_mae
if delta > 0:
    print(
        f"  → Dropping those features cost +${delta:.1f}k MAE — they were worth collecting."
    )
else:
    print(f"  → No cost — those features weren't contributing signal worth keeping.")

---

## Part 2 — Gradient Descent from Scratch

Part 1 called `sklearn.LinearRegression().fit()` — but what actually happened inside that call? Knowing this matters because the same mechanism trains every model from logistic regression to GPT: **gradient descent**, a 10-line loop that works when there's no closed-form formula (which is almost always).

Linear regression happens to have a closed-form solution — but every other model in this notebook (and every neural network you'll encounter) doesn't. The universal training algorithm is gradient descent:

$$w \leftarrow w - \alpha \cdot \frac{\partial \text{MSE}}{\partial w} = w - \alpha \cdot \frac{2}{n} X^T(Xw - y)$$

Let's implement this manually and verify it converges to the same solution as sklearn.


> **Intuition first:** Imagine standing on a hilly loss surface wearing a blindfold. You can only feel the slope under your feet. You take one step in the direction that goes most steeply downhill, then feel the slope again and repeat. That's gradient descent. The gradient tells you the slope direction; the learning rate controls your step size.


![MSE loss landscape showing a gradient descent path spiralling down to the minimum](images/regression-loss-landscape.png)


In [ ]:
#  Part 2: Linear regression via gradient descent 
X_gd = np.column_stack([X_train_s, np.ones(len(X_train_s))])  # add bias column
n_features = X_gd.shape[1]

weights = np.zeros(n_features)  # initialise at zero
lr_rate = 0.01
losses = []

print("Manual gradient descent on California Housing:")
for epoch in range(200):
    pred = X_gd @ weights
    residuals = pred - y_train
    mse = (residuals**2).mean()
    losses.append(mse)
    # Gradient of MSE w.r.t. weights: 2/n * X^T (Xw - y)
    grad = 2 / len(y_train) * X_gd.T @ residuals
    weights = weights - lr_rate * grad
    if (epoch + 1) % 50 == 0:
        mae_gd = np.abs(pred - y_train).mean() * 100
        print(f"  epoch {epoch+1:3d}: MSE={mse:.4f}  MAE=${mae_gd:.1f}k")

# Compare to sklearn
sklearn_weights = np.append(lr.coef_, lr.intercept_)
max_diff = np.abs(weights - sklearn_weights).max()
print(f"\n  Max weight difference vs. sklearn: {max_diff:.6f}")
print(
    f"   Converged to same solution"
    if max_diff < 0.01
    else f"  → Needs more iterations"
)

In [ ]:
#  Part 2: Loss convergence plot 
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, "steelblue", lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Gradient descent convergence on California Housing")
plt.tight_layout()
plt.show()
print(f"MSE dropped from {losses[0]:.2f} to {losses[-1]:.4f} in 200 epochs.")
print("→ The SAME gradient descent loop trains every neural network.")

#### #### Predict first: learning rate too high vs. too low

The gradient descent loop above used `lr_rate = 0.01` and converged smoothly. Now imagine rerunning that **exact same loop, on the exact same SmartVal data** — changing only the learning rate.

**If `lr_rate` is 100× too large (`1.0`):**
A) Converges faster to a lower loss
B) Diverges/oscillates — loss explodes to huge values or `NaN`
C) Barely moves — loss stays roughly flat after many epochs

**If `lr_rate` is 100× too small (`0.0001`):**
A) Converges faster to a lower loss
B) Diverges/oscillates — loss explodes to huge values or `NaN`
C) Barely moves — loss stays roughly flat after many epochs

Make your prediction for both cases, then run the cell below to see all three learning rates plotted together.


In [ ]:
#  Part 2: Learning rate too high vs. good vs. too low 
# Reuses the SAME gradient descent update rule, the SAME feature matrix (X_gd)
# and the SAME target (y_train) as the manual GD loop above — only `lr` changes.
def gd_run(lr, steps=200):
    w = np.zeros(n_features)
    run_losses = []
    for _ in range(steps):
        pred = X_gd @ w
        residuals = pred - y_train
        mse = (residuals**2).mean()
        if not np.isfinite(mse) or mse > 1e8:
            run_losses.extend([np.nan] * (steps - len(run_losses)))
            break
        run_losses.append(mse)
        grad = 2 / len(y_train) * X_gd.T @ residuals
        w = w - lr * grad
    return run_losses, w


lr_too_high = lr_rate * 100  # 1.0    — 100x too large
lr_good = lr_rate  # 0.01   — the rate used above
lr_too_low = lr_rate / 100  # 0.0001 — 100x too small

fig, ax = plt.subplots(figsize=(9, 4))
for rate, label, color in [
    (lr_too_high, f"lr={lr_too_high:g} (100x too high)", "#e74c3c"),
    (lr_good, f"lr={lr_good:g} (good — used above)", "#2ecc71"),
    (lr_too_low, f"lr={lr_too_low:g} (100x too low)", "#6b9ac4"),
]:
    run_losses, final_w = gd_run(rate)
    ax.plot(run_losses, label=label, color=color, linewidth=2)
    if np.isfinite(run_losses[-1]):
        final_mae = np.abs(X_gd @ final_w - y_train).mean() * 100
        print(
            f"  {label:32s} final MSE={run_losses[-1]:.4g}  final train MAE=${final_mae:.1f}k"
        )
    else:
        print(f"  {label:32s} final MSE=NaN (diverged)")

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss (log scale)")
ax.set_title("SmartVal: same gradient descent loop, three learning rates")
ax.legend()
plt.tight_layout()
plt.show()

#### What just happened — and what's missing

- **`lr=1.0` (100× too high)** — the loss curve shoots upward and diverges within the first few epochs (huge values or `NaN`). Each step overshoots the minimum by so much that the error compounds instead of shrinking.
- **`lr=0.0001` (100× too low)** — the loss curve is almost flat across all 200 epochs. It's moving in the right direction, just far too slowly to be usable within this iteration budget.
- **`lr=0.01` (the rate used above)** — this is the only curve that keeps dropping steadily, and it's the only one that gets SmartVal's MAE under the **$40k target**.

→ The learning rate isn't a minor tuning knob — pick it wrong and you either blow up training or waste your entire iteration budget without converging.

**What's missing:** Both Part 1 and Part 2 minimised MSE — but is MSE actually the right loss for SmartVal? The California Housing dataset has ~200 price-capped districts that create systematically huge errors no model can explain. MSE forces the model to spend capacity chasing those unchaseable outliers. Part 3 measures what happens if you choose a different loss.


#### #### Your turn — epoch budget

The loop above used 200 epochs with `lr=0.01`. What if you change the epoch budget? More epochs should converge further — but past a certain point, diminishing returns set in.


In [ ]:
#  #### Your turn: epoch sensitivity 
# # CHANGE n_epochs_yt to 50, 100, 500, or 2000 — observe when improvement plateaus
n_epochs_yt = 200  # # CHANGE ME

w_yt = np.zeros(n_features)
yt_losses = []
for _ in range(n_epochs_yt):
    pred_yt = X_gd @ w_yt
    res_yt = pred_yt - y_train
    yt_losses.append((res_yt**2).mean())
    w_yt -= 0.01 * (2 / len(y_train)) * X_gd.T @ res_yt

final_mae_yt = np.abs(X_gd @ w_yt - y_train).mean() * 100
improvement = (yt_losses[0] - yt_losses[-1]) / yt_losses[0] * 100
print(f"After {n_epochs_yt} epochs: train MAE = ${final_mae_yt:.1f}k")
print(
    f"  MSE: {yt_losses[0]:.4f} → {yt_losses[-1]:.4f}  ({improvement:.1f}% reduction)"
)
print("→ At what epoch count does more training stop meaningfully reducing MAE?")

---

## Part 3 — Loss Functions: MSE vs. MAE

Gradient descent minimises _something_ — but what you choose as "something" shapes the weights you get. SmartVal's California data has an artifact: ~200 districts are price-capped at $500k, creating errors that reflect the data collection process, not real price variation. If MSE is the loss, those outliers attract disproportionate gradient and distort every weight the model learns.

MSE ($\sum e_i^2$) penalises large errors heavily — one $500k-error district swamps ten $50k errors. MAE ($\sum |e_i|$) treats all errors equally. Which should SmartVal use?

#### #### Predict first

3 of SmartVal's districts have true values capped at $500k in the dataset (a data artifact causing extreme errors). Will switching from MSE to MAE improve or worsen the overall MAE on normal districts?

1. **Improve** — MAE is less distorted by the $500k outlier districts
2. **Worsen** — MAE-trained models are less precise in general
3. **No change** — both losses lead to the same weights for linear regression


In [ ]:
#  Part 3: MSE vs. MAE loss comparison 
from sklearn.linear_model import HuberRegressor

# Find outlier districts (capped at $500k in the dataset)
outlier_mask = y_val >= 4.99  # ≥$499k (the cap artifact)
n_outliers = outlier_mask.sum()
print(f"Districts with price ≥$499k (likely capped): {n_outliers}/{len(y_val)}")

# MSE-trained model (standard sklearn LinearRegression)
mse_val_mae = mean_absolute_error(y_val[~outlier_mask], val_pred[~outlier_mask]) * 100

# Huber regression (robust to outliers, approximates MAE for large errors)
huber = HuberRegressor(epsilon=1.35, max_iter=500)
huber.fit(X_train_s, y_train)
huber_pred = huber.predict(X_val_s)
huber_val_mae = (
    mean_absolute_error(y_val[~outlier_mask], huber_pred[~outlier_mask]) * 100
)

print(f"\nPerformance on NON-outlier districts:")
print(f"  MSE-trained model MAE:   ${mse_val_mae:.1f}k")
print(f"  Robust model MAE:        ${huber_val_mae:.1f}k")
print()
if huber_val_mae < mse_val_mae:
    print(
        "→ Robust loss improved MAE on normal districts by ignoring outlier distortion"
    )
    print("  Prediction 1 is confirmed.")
else:
    print(
        "→ MSE model performed similarly — outlier count may be too small to matter here"
    )

print()
print("Rule of thumb:")
print("  MSE when you care about large errors (safety-critical predictions)")
print("  MAE or Huber when your dataset has noisy/capped outliers")

#### What just happened — and what's missing

The Huber model improved MAE on non-outlier districts by down-weighting the capped $500k artifacts, confirming that **the loss function shapes what the model learns** — not just how it scores. Choosing MSE when outliers are present forces the model to explain noise it can never fix.

**What's missing:** Both models above used all 8 features. Carrying irrelevant features forces weights onto noise — and on new districts, those weights hurt. Part 4 introduces a penalty that makes the model justify every feature it keeps, and collapses truly irrelevant ones to exactly zero.

---

#### #### Your turn — Huber epsilon

The `epsilon` parameter controls where Huber transitions from MSE-like (for small errors) to MAE-like (for large errors). A smaller epsilon makes the loss more MAE-like overall.


In [ ]:
#  #### Your turn: Huber epsilon sensitivity 
# epsilon controls where Huber transitions from MSE-like to MAE-like.
# # CHANGE epsilon_yt to 1.0 (more MAE-like) or 2.0 (more MSE-like) and compare
epsilon_yt = 1.35  # # CHANGE ME

huber_yt = HuberRegressor(epsilon=epsilon_yt, max_iter=500)
huber_yt.fit(X_train_s, y_train)
h_pred_yt = huber_yt.predict(X_val_s)
h_mae_yt = mean_absolute_error(y_val[~outlier_mask], h_pred_yt[~outlier_mask]) * 100

print(f"epsilon={epsilon_yt}: non-outlier val MAE = ${h_mae_yt:.1f}k")
print(
    f"  (MSE model: ${mse_val_mae:.1f}k | default epsilon=1.35: ${huber_val_mae:.1f}k)"
)
if h_mae_yt < huber_val_mae:
    print(
        f"  → epsilon={epsilon_yt} beats default by ${huber_val_mae - h_mae_yt:.1f}k — suits this dataset better."
    )
elif h_mae_yt > huber_val_mae:
    print(
        f"  → Worse than default by ${h_mae_yt - huber_val_mae:.1f}k — 1.35 is a good choice here."
    )
else:
    print("  → Same as default.")

---

## Part 4 — Regularisation: Preventing Overfitting and Selecting Features

SmartVal pays to collect all 8 census features per district — but data acquisition has a cost per variable. If 3 of those 8 features contribute near-zero signal, every unnecessary variable increases acquisition costs without improving predictions. Regularisation is a penalty added to the loss that forces the model to justify every feature it uses: large weights are expensive, so the model only keeps them when they genuinely help.

**Ridge** adds $\lambda \sum w_i^2$ to the loss — all weights shrink toward zero but none reach it exactly. Stabilises weights when features are correlated.  
**Lasso** adds $\lambda \sum |w_i|$ — some weights become _exactly_ zero, directly eliminating features. SmartVal can use Lasso to identify which census variables are worth the collection cost.


![LASSO vs Ridge: Lasso zeroes out irrelevant features, Ridge shrinks all](images/lasso-ridge-coefficients.png)


#### #### Predict first

As regularisation strength α increases from 0.001 to 10.0 in the Lasso sweep, predict what happens to val MAE:

1. **Only improves** — stronger regularisation always helps generalisation
2. **U-shape** — too little overfits; too much zeroes out useful features and underfits
3. **Stays flat** — regularisation changes weights but not prediction quality

And at the highest α tested (10.0), how many of SmartVal's 8 features will Lasso keep?

A) All 8 — Lasso won't zero any features at this α  
B) 4–6 features remain  
C) Fewer than 4 — the high penalty collapses most features to zero


In [ ]:
#  Part 4: Ridge vs. Lasso regularisation 
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

ridge_maes, lasso_maes = [], []
ridge_nonzero, lasso_nonzero = [], []

for alpha in alphas:
    r = Ridge(alpha=alpha)
    r.fit(X_train_s, y_train)
    l = Lasso(alpha=alpha, max_iter=5000)
    l.fit(X_train_s, y_train)

    ridge_maes.append(mean_absolute_error(y_val, r.predict(X_val_s)) * 100)
    lasso_maes.append(mean_absolute_error(y_val, l.predict(X_val_s)) * 100)
    ridge_nonzero.append((r.coef_ != 0).sum())
    lasso_nonzero.append((l.coef_ != 0).sum())

print("α       Ridge MAE   Ridge #feat  Lasso MAE   Lasso #feat")
for a, rm, rf, lm, lf in zip(
    alphas, ridge_maes, ridge_nonzero, lasso_maes, lasso_nonzero
):
    print(f"  {a:.3f}   ${rm:.1f}k     {rf}/8          ${lm:.1f}k     {lf}/8")

# Best regularisation
best_alpha_r = alphas[np.argmin(ridge_maes)]
best_alpha_l = alphas[np.argmin(lasso_maes)]
print(f"\n  Best Ridge α: {best_alpha_r}  |  Best Lasso α: {best_alpha_l}")

# Show which features Lasso zeroes out at best alpha
l_best = Lasso(alpha=best_alpha_l, max_iter=5000)
l_best.fit(X_train_s, y_train)
print("\nLasso feature weights at best α:")
for name, w in zip(feature_names, l_best.coef_):
    status = "ZERO" if w == 0 else f"{w:+.4f}"
    print(f"  {name:12s}: {status}")

#### What just happened — and what's missing

The α sweep shows the U-shape: at very low α, regularisation is too weak to help; at very high α, Lasso eliminates useful features and MAE climbs back up. The feature table reveals which census variables survive — a coefficient at exactly zero means the training data found that variable not worth the weight beyond what the kept features already explain.

**What's missing:** Ridge and Lasso both output a number — a predicted house price. But SmartVal's regulatory team also needs a binary flag: "high-value district (>$300k)?" Outputting a price and thresholding it is possible, but Part 5 shows the proper tool: a model designed from the start to output a calibrated probability.

---

#### #### Your turn — fine-grained alpha

The sweep above tested 5 α values. Is there a better value in between? Try narrowing the search.


In [ ]:
#  #### Your turn: fine-grained alpha search 
# The sweep above tested 5 α values. Is there a better one in between?
# # CHANGE alpha_yt to any positive number (e.g. 0.005, 0.03, 0.3, 3.0)
alpha_yt = 0.01  # # CHANGE ME

ridge_yt = Ridge(alpha=alpha_yt)
ridge_yt.fit(X_train_s, y_train)
lasso_yt = Lasso(alpha=alpha_yt, max_iter=5000)
lasso_yt.fit(X_train_s, y_train)

ridge_mae_yt = mean_absolute_error(y_val, ridge_yt.predict(X_val_s)) * 100
lasso_mae_yt = mean_absolute_error(y_val, lasso_yt.predict(X_val_s)) * 100
n_nonzero_yt = (lasso_yt.coef_ != 0).sum()

print(f"α = {alpha_yt}:")
print(
    f"  Ridge val MAE: ${ridge_mae_yt:.1f}k  |  Lasso val MAE: ${lasso_mae_yt:.1f}k  ({n_nonzero_yt}/8 features kept)"
)
print(
    f"  (Sweep best — Ridge α={best_alpha_r}: ${min(ridge_maes):.1f}k  |  Lasso α={best_alpha_l}: ${min(lasso_maes):.1f}k)"
)

---

## Part 5 — Binary Classification: "High-Value District?"

SmartVal wants to flag districts where median house value exceeds $300k. This converts the regression problem into binary classification. The model outputs a probability: $\hat{p} = \sigma(w^T x + b)$ where $\sigma$ is the sigmoid function. Training minimises Binary Cross-Entropy loss.


#### #### Predict first

SmartVal's logistic regression will classify districts as "high value" (>$300k) or not. About 40% of validation districts are high-value. Predict the val accuracy:

1. **~50–60%** — barely above random chance; the linear boundary struggles to separate the classes
2. **~70–80%** — decent separation but many boundary districts are misclassified
3. **~85%+** — the linear boundary cleanly captures the income + location signal

Also predict: with the default 0.5 threshold, will the model produce more **false positives** (flagging a non-high-value district) or more **false negatives** (missing a truly high-value district)? Check the confusion matrix to verify.


In [ ]:
#  Part 5: Binary classification 
THRESHOLD = 3.0  # $300k in $100k units
y_train_bin = (y_train >= THRESHOLD).astype(int)
y_val_bin = (y_val >= THRESHOLD).astype(int)

clf = LogisticRegression(random_state=42, max_iter=500)
clf.fit(X_train_s, y_train_bin)
val_prob = clf.predict_proba(X_val_s)[:, 1]
val_pred_bin = (val_prob >= 0.5).astype(int)

acc = accuracy_score(y_val_bin, val_pred_bin)
cm = confusion_matrix(y_val_bin, val_pred_bin)

print(f"High-value district (>${THRESHOLD*100:.0f}k) classification:")
print(f"  Val accuracy: {acc:.1%}")
print()
print("Confusion matrix:")
print(f"          Pred: low  Pred: high")
print(f"  True: low   {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"  True: high  {cm[1,0]:5d}    {cm[1,1]:5d}")
print()
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(f"  Precision: {precision:.1%}  (of flagged high-value, how many truly are?)")
print(f"  Recall:    {recall:.1%}   (of truly high-value, how many did we catch?)")

In [ ]:
#  Part 5: Sigmoid function and probability calibration 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Sigmoid
z = np.linspace(-6, 6, 200)
sigma = 1 / (1 + np.exp(-z))
ax1.plot(z, sigma, "steelblue", lw=2)
ax1.axhline(0.5, color="coral", ls="--", lw=1, label="Decision boundary (p=0.5)")
ax1.axvline(0, color="gray", lw=0.5)
ax1.set_xlabel("z = w·x + b")
ax1.set_ylabel("σ(z) = P(high value)")
ax1.set_title("Sigmoid: linear score → probability")
ax1.legend()

# Predicted probabilities distribution
ax2.hist(
    val_prob[y_val_bin == 0],
    bins=30,
    alpha=0.6,
    color="steelblue",
    label="True: low value",
)
ax2.hist(
    val_prob[y_val_bin == 1],
    bins=30,
    alpha=0.6,
    color="coral",
    label="True: high value",
)
ax2.axvline(0.5, color="black", ls="--", lw=1.5)
ax2.set_xlabel("Predicted P(high value)")
ax2.set_ylabel("Count")
ax2.set_title("Predicted probability distributions")
ax2.legend()

plt.tight_layout()
plt.show()

#### What just happened — and what's missing

The probability distribution plot shows the two classes largely separating — high-income, well-located districts cluster near P=1 and low-value districts cluster near P=0. The sigmoid curve is the mathematical mechanism converting the same linear score ($w^T x + b$) from Parts 1–4 into a calibrated probability.

**What's missing:** All 6 Parts so far trained on 13,200 samples. SmartVal's pilot deployment may start with a single region — maybe 50 labelled districts. Part 6 asks: what actually happens if you train on that few samples?

---

#### #### Your turn — decision threshold

The default threshold is 0.5. Lowering it flags more districts as high-value (higher recall, lower precision). SmartVal's regulators may prefer different tradeoffs depending on which error is costlier.


In [ ]:
#  #### Your turn: decision threshold sensitivity 
# Default threshold is 0.5: predict high-value when P(high) >= 0.5.
# # CHANGE threshold_yt to 0.3 (flag more districts) or 0.7 (only confident predictions)
threshold_yt = 0.5  # # CHANGE ME

pred_yt = (val_prob >= threshold_yt).astype(int)
cm_yt = confusion_matrix(y_val_bin, pred_yt)
tn_yt, fp_yt, fn_yt, tp_yt = cm_yt.ravel()
prec_yt = tp_yt / (tp_yt + fp_yt) if (tp_yt + fp_yt) > 0 else 0
rec_yt = tp_yt / (tp_yt + fn_yt) if (tp_yt + fn_yt) > 0 else 0
acc_yt = accuracy_score(y_val_bin, pred_yt)

print(f"Threshold = {threshold_yt}:")
print(f"  Accuracy: {acc_yt:.1%}  |  Precision: {prec_yt:.1%}  |  Recall: {rec_yt:.1%}")
print(f"  False positives: {fp_yt}  |  False negatives: {fn_yt}")
print(
    "→ Lowering threshold: recall↑ (catches more high-value) + precision↓ (more false alarms)"
)
print(
    "→ Raising threshold:  precision↑ (fewer false alarms) + recall↓ (misses more high-value)"
)

---

## Part 6 — Overfitting: When the Model Memorises Instead of Learning

SmartVal's marketing team proposes training on just the 50 districts they know best. This is dangerous: a model can memorise 50 data points perfectly but fail completely on new districts.

#### #### Predict first

Training a linear regression on only 50 samples from California Housing, what will the validation MAE be compared to the full 13,000-sample model?

1. **Better** — less data makes the model "focus" on what matters
2. **Worse** — overfitting: the model memorises training noise and fails on new data
3. **Same** — linear regression doesn't overfit regardless of sample size


![Overfitting: training loss falls while validation loss forms a U-shape, with the early-stop sweet spot marked](images/overfitting-train-val-curves.png)


In [ ]:
#  Part 6: Overfitting demonstration 
sample_sizes = [20, 50, 100, 200, 500, 1000, 5000, len(X_train_s)]
train_maes_ov, val_maes_ov = [], []

for n in sample_sizes:
    idx = np.random.choice(len(X_train_s), n, replace=False)
    Xn, yn = X_train_s[idx], y_train[idx]
    m = LinearRegression()
    m.fit(Xn, yn)
    train_maes_ov.append(mean_absolute_error(yn, m.predict(Xn)) * 100)
    val_maes_ov.append(mean_absolute_error(y_val, m.predict(X_val_s)) * 100)
    print(
        f"  n={n:6d}: train MAE=${train_maes_ov[-1]:.1f}k  val MAE=${val_maes_ov[-1]:.1f}k"
    )

print()
print("Prediction check:")
print(
    f"  n=50 val MAE: ${val_maes_ov[1]:.1f}k  vs.  n=full val MAE: ${val_maes_ov[-1]:.1f}k"
)
if val_maes_ov[1] > val_maes_ov[-1]:
    print(
        "  → Answer 2 confirmed: fewer samples = worse generalisation (even for linear models)"
    )

In [ ]:
#  Part 6: Learning curve 
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    range(len(sample_sizes)),
    train_maes_ov,
    "o-",
    color="steelblue",
    lw=2,
    label="Train MAE",
)
ax.plot(
    range(len(sample_sizes)), val_maes_ov, "s-", color="coral", lw=2, label="Val MAE"
)
ax.axhline(40, color="green", ls="--", lw=1.5, label="Target: $40k MAE")
ax.set_xticks(range(len(sample_sizes)))
ax.set_xticklabels([str(n) for n in sample_sizes], rotation=30)
ax.set_xlabel("Training samples")
ax.set_ylabel("MAE ($k)")
ax.set_title("Learning curve: more data → better generalisation")
ax.legend()
plt.tight_layout()
plt.show()

target_n = next((n for n, m in zip(sample_sizes, val_maes_ov) if m <= 40), None)
if target_n:
    print(f"SmartVal needs at least ~{target_n:,} samples to hit the $40k MAE target.")

#### What just happened — and what's missing

The learning curve makes overfitting _visible_: at n=20, train MAE is near zero (the model memorised every point) while val MAE is massive — it answered "what predicted those 20 districts" not "what predicts all California districts." The gap closes as n grows, and the curve shows exactly how many samples SmartVal needs to cross the $40k threshold.

**What's missing:** Everything in this notebook used 8 features, CPU-seconds of training, and a controlled experimental loop. The Summary below maps each toy setting to its production equivalent before giving SmartVal's final model recommendation.

---

#### #### Your turn — regularisation payoff at small n

Regularisation should pay off most when data is scarce — the penalty prevents overfitting to a tiny sample. Test that hypothesis directly.


In [ ]:
#  #### Your turn: regularisation payoff at small sample size 
# # CHANGE n_small to 20, 50, 100, or 200 — does Ridge help more when data is scarce?
n_small = 50  # # CHANGE ME

np.random.seed(42)
idx_small = np.random.choice(len(X_train_s), n_small, replace=False)
Xs_yt, ys_yt = X_train_s[idx_small], y_train[idx_small]

lr_small = LinearRegression()
lr_small.fit(Xs_yt, ys_yt)
ridge_small = Ridge(alpha=best_alpha_r)
ridge_small.fit(Xs_yt, ys_yt)

mae_plain_yt = mean_absolute_error(y_val, lr_small.predict(X_val_s)) * 100
mae_ridge_yt = mean_absolute_error(y_val, ridge_small.predict(X_val_s)) * 100

print(f"n={n_small}:")
print(f"  Plain LR val MAE:  ${mae_plain_yt:.1f}k")
print(f"  Ridge val MAE:     ${mae_ridge_yt:.1f}k")
if mae_ridge_yt < mae_plain_yt:
    print(
        f"  → Ridge saved ${mae_plain_yt - mae_ridge_yt:.1f}k — regularisation pays off most when data is scarce."
    )
else:
    print(
        f"  → Ridge didn't help at n={n_small} — try n=20 to see a more extreme case."
    )

---

## Summary and Closing Decision

### Toy → Production Bridge

Before reading SmartVal's final recommendation, here's how every setting in this notebook maps to production-scale ML:

| Concept | This notebook | Production |
|---|---|---|
| Training samples | 13,200 districts | 100k–10M rows; data curation is half the job |
| Features | 8 numeric, pre-cleaned | 100s–1000s; requires pipeline of encoders, scalers, imputers |
| Linear regression | 1 call, instant fit | Still often the baseline; Stochastic GD for 10M+ rows |
| Gradient descent | 200 epochs, full-batch | Mini-batch SGD (batch 32–512); Adam/AdamW replace vanilla GD |
| Learning rate | Fixed 0.01 | Warmup + decay schedule; found by LR range test or sweep |
| Regularisation α | 5-value manual sweep | Cross-validated grid/random search; α often 0.0001–1.0 |
| Train/val split | Random 80/20 | Time-stratified if temporal; k-fold CV for small datasets |
| Overfitting check | 8 sample sizes, same model | Early stopping on val loss per epoch; monitor train/val gap |
| Regulatory MAE target | $40k (SmartVal) | Domain-defined SLA; always evaluated on held-out test set |

---

| Part | SmartVal question           | Answer                                                           |
| ---- | --------------------------- | ---------------------------------------------------------------- |
| 1    | Baseline MAE?               | Linear regression — run the Part 1 cell to see val MAE           |
| 2    | Can we train manually?      | Yes — gradient descent matches sklearn                           |
| 3    | MSE vs. MAE for outliers?   | Huber/robust loss is better when caps exist                      |
| 4    | Which features matter?      | Lasso zeroes out irrelevant ones                                 |
| 5    | Can we classify high-value? | Logistic regression with accuracy/precision/recall printed above |
| 6    | Is 50 samples enough?       | No — see learning curve for minimum sample needed                |

### Key insights to keep

- **Part 1:** A linear model with 8 census features can get within $40k MAE on 20,000 districts — the baseline is often competitive with far more complex alternatives, and always worth establishing first.
- **Part 2:** Every neural network in every framework is trained by gradient descent; the 10-line loop above _is_ that engine, just with narrower vectors.
- **Part 3:** The loss function is a design decision, not a default — MSE and MAE train measurably different models on the same data when outliers are present.
- **Part 4:** Lasso doesn't just regularise, it performs feature selection; a coefficient at exactly zero means the training data found that feature not worth the weight.
- **Part 5:** Classification and regression reuse the same linear scoring ($w^T x + b$); the only difference is wrapping that score in a sigmoid and measuring calibrated probability instead of value.
- **Part 6:** A model trained on 50 samples isn't just imprecise — it's answering a different (smaller) question; sample count is a hard constraint, not a tuning knob.


In [ ]:
#  Closing Decision — SmartVal's recommendation 
print("=" * 55)
print("  CLOSING DECISION — SmartVal AI Model Selection")
print("=" * 55)
print()
print(f"  Baseline linear regression (full dataset):")
print(
    f"    Val MAE: ${val_mae:.1f}k  {' TARGET MET' if val_mae <= 40 else ' Below target'}"
)
print()
print(f"  High-value district classifier:")
print(
    f"    Accuracy: {acc:.1%}  |  Precision: {precision:.1%}  |  Recall: {recall:.1%}"
)
print()
target_n_str = f"{target_n:,}" if target_n else "more than tested"
print(f"  Minimum training data for $40k target: ~{target_n_str} samples")
print()
print("  RECOMMENDATION:")
print(f"  → Use Ridge regularisation (α={best_alpha_r}) for the regression model")
print("  → Lasso can reduce feature acquisition costs by zeroing irrelevant features")
print(
    "  → Do NOT train on fewer than 1,000 samples — overfitting will disqualify approval"
)
print()
print("  NEXT STEP: The linear model assumes a linear relationship between")
print("  features and price. To capture nonlinear patterns (e.g., interaction")
print("  between income AND location), we need neural networks — next chapter.")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- Linear regression — closed-form + gradient descent; matched results
- MSE loss and gradient derivation — implemented from scratch and verified
- Ridge vs. Lasso regularisation — coefficient comparison at 5 α values
- Binary classification — logistic regression, sigmoid, confusion matrix
- Overfitting — learning curves showing training/val divergence

### Tier 2 — Explained but Not Fully Implemented

- **Huber loss** — shown as a robust alternative; loss function explained but custom training loop not built
- **Polynomial features** — mentioned in the "nonlinear patterns" pointer; `sklearn.preprocessing.PolynomialFeatures` exists but not used

### Tier 3 — Named but Out of Scope

- **SVMs** — support vector machines for classification; kernel trick enables nonlinear boundaries; standard in notes/01-ml/02-classification but more advanced than needed here
- **Tree-based models** — Random Forest, Gradient Boosting; strong baselines for tabular data; covered in notes/01-ml/08-ensemble-methods
- **Neural network classifiers** — the natural extension of logistic regression; covered starting in the next chapter


---

## When to Use What — ML Basics

| Situation                                     | Choose                              | Reason                             |
| --------------------------------------------- | ----------------------------------- | ---------------------------------- |
| Continuous target, linear relationship likely | Linear regression + Ridge           | Fast, interpretable, good baseline |
| Continuous target, want feature selection     | Lasso                               | Zeroes out irrelevant features     |
| Continuous target, noisy/capped outliers      | Huber regression                    | Robust to extreme values           |
| Binary outcome                                | Logistic regression                 | Outputs calibrated probability     |
| Training data < 1,000 samples                 | Add regularisation + watch val loss | Risk of overfitting is high        |

→ **Next:** `learning/genai-prerequisites/02-neural-networks/` — when the linear relationship assumption breaks (spiral datasets, XOR), we need hidden layers and nonlinear activations.
